In [1]:
from dotenv import load_dotenv
from langfuse import get_client
from dataclasses import dataclass
import pandas as pd

In [2]:
dataset = pd.read_csv('questions_dataset.csv', header=0)

dev_dataset_name = "20250520_clean_dev_dataset"
test_dataset_name = "20250520_test_dataset"

In [3]:
load_dotenv()
langfuse = get_client()

In [4]:
# create datasets in Langfuse

langfuse.create_dataset(
    name=dev_dataset_name,
    description="Dataset containg dev questions",
    metadata={
        "author": "Saskya",
        "date": "2026-05-20",
        "type": "dev"
    }
)

langfuse.create_dataset(
    name=test_dataset_name,
    description="Dataset containg test questions",
    metadata={
        "author": "Saskya",
        "date": "2026-05-20",
        "type": "test"
    }
)

Dataset(id='cmpdor8z20029nw07vwoxr9be', name='20250520_test_dataset', description='Dataset containg test questions', metadata={'date': '2026-05-20', 'type': 'test', 'author': 'Saskya'}, input_schema=None, expected_output_schema=None, project_id='cmmvqmjqn0007nw07hbqsche1', created_at=datetime.datetime(2026, 5, 20, 6, 33, 30, 927000, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2026, 5, 20, 6, 33, 30, 927000, tzinfo=datetime.timezone.utc))

In [5]:
# populate created datasets

@dataclass
class DatasetItem:
    question: str
    answer: str
    doc_id: str
    paragraph: str

def add_item_in_dataset(langfuse_client, dataset_name, item):
    langfuse_client.create_dataset_item(
            dataset_name=dataset_name,
            input={
                "query": item.question
            },
            expected_output={
                "answer": item.answer
            },
            metadata={
                "expected_doc_id": item.doc_id,
                "expected_paragraph": item.paragraph
            }
        )

for _, row in dataset.iterrows():
    item = DatasetItem(
        question=row["Question"],
        answer=row["Réponse"],
        doc_id=row["Id document"],
        paragraph=(row["Article / Paragraphe"] if pd.notna(row["Article / Paragraphe"]) else "")
    )
    if row['Dataset'] == "dev":
        add_item_in_dataset(langfuse, dev_dataset_name, item)
    elif row['Dataset'] == "test":
        add_item_in_dataset(langfuse, test_dataset_name, item)